# T35 — Efficiency of all four backbones, measured in one session

Parameters, GFLOPs, checkpoint size, and CPU and GPU latency for the baseline (A0) and the selected arm
of each backbone, using the trained seed-42 checkpoints. All eight models are timed in this one
session on one machine, so the latencies can be compared with each other — which the existing
per-backbone efficiency files cannot.

**Expected run time on one T4:** about 10–15 minutes (most of it is batch-1 CPU timing of ViT-Base).

**Before running (Kaggle sidebar):**
1. Session options → Accelerator → **GPU T4** (one is enough). Internet → **On**.
2. Add input → the primary dataset `tawsifurrahman/covid19-radiography-database`.
3. Add input → every dataset that holds the trained `.pt` checkpoints. The notebook searches all of
   `/kaggle/input` for `.pt` files and reads the backbone, arm and seed stored inside each one, so folder
   names do not matter. The seed-42 checkpoints are already in `uom230334h/t28-checkpoints`,
   `uom230334h/t28-vitbase-checkpoints` and `uom230334h/t28-efficientnet-checkpoints`. The seed 123 and
   2026 checkpoints (A0 and the selected arm of each backbone) still have to be uploaded as a dataset.
4. Run All. The inventory cell prints which checkpoints were found and which core-grid ones are missing;
   the run continues with whatever is present.

Only the eight seed-42 checkpoints (A0 and the selected arm) are needed here.

**Output:** `/kaggle/working/T35_efficiency.zip` with `efficiency_all.csv`, `efficiency_overhead.csv`
and `efficiency_hardware.json`.

In [ ]:
import os, sys, json, time, math, hashlib, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/Ravindu-Pathirana/Chest-X-ray-Disease-Detection.git"
REPO_BRANCH = "t26-vit-kaggle-results"   # holds the ViT-Base results; falls back to main if it has been merged and deleted

# Everything below is found automatically under /kaggle/input. The environment variables only exist
# so the notebook can also be dry-run off Kaggle.
DATA_ROOT = Path(os.environ.get("TRUST_DATA_ROOT", "/kaggle/input"))   # searched for COVID-19_Radiography_Dataset
CKPT_ROOT = Path(os.environ.get("TRUST_CKPT_ROOT", "/kaggle/input"))   # searched recursively for *.pt
WORK      = Path(os.environ.get("TRUST_WORK", "/kaggle/working"))
REPO      = Path(os.environ.get("TRUST_REPO", str(WORK / "Chest-X-ray-Disease-Detection")))
OUT       = WORK / "T35_efficiency"

BATCH_SIZE  = 64
NUM_WORKERS = int(os.environ.get("TRUST_WORKERS", "4"))
DRY_RUN     = os.environ.get("TRUST_DRY_RUN", "0") == "1"   # set to True for a 2-checkpoint trial run

CPU_RUNS, GPU_RUNS = 50, 100      # batch-1 timing repeats (the harness adds its own warm-up)
ALL_SEED42_ARMS = False           # True: benchmark every seed-42 arm instead of A0 + selected only

OUT.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT, "| dry run:", DRY_RUN)

In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    if subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_BRANCH]).returncode != 0:
        print(f"branch {REPO_BRANCH} not found -> staying on the default branch")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo commit:", COMMIT)

for module, package in [("timm", "timm"), ("pytorch_grad_cam", "grad-cam")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if str(WORK).startswith("/kaggle") and DEVICE.type != "cuda":
    raise RuntimeError("No GPU. In the Kaggle sidebar set Session options -> Accelerator -> GPU T4, then Run All again.")

try:
    import fvcore  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fvcore"])

In [ ]:
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
SEEDS = [42, 123, 2026]
ARTIFACT = {"densenet121": "T18_lung_attention", "resnet50": "T23_resnet50_lung_attention",
            "efficientnet_b0": "T25_efficientnet_b0_lung_attention",
            "vit_base_patch16_224": "T26_vit_base_lung_attention"}
BASE_ARM = {"densenet121": "A0_vanilla", "resnet50": "A0_vanilla", "efficientnet_b0": "A0",
            "vit_base_patch16_224": "A0_vanilla"}
SELECTED_ARM = {"densenet121": "A2_full", "resnet50": "A3_multiply", "efficientnet_b0": "A3",
                "vit_base_patch16_224": "A3_multiply"}
SKIP_DIRS = {"images", "masks", ".venv", ".git", "__pycache__", "stage_2_train_images", "stage_2_test_images"}


def find_data_dir(root):
    for d, subdirs, _ in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        p = Path(d)
        if (p / "COVID" / "images").is_dir() and (p / "Viral Pneumonia" / "masks").is_dir():
            return p
    raise FileNotFoundError(f"COVID-19_Radiography_Dataset (with images/ and masks/) not found under {root}. "
                            "Attach the dataset tawsifurrahman/covid19-radiography-database.")


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def sha256_12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


def reference_dir(backbone, arm, seed):
    """Folder in the repo that holds this run's committed results."""
    base = REPO / "artifacts" / ARTIFACT[backbone]
    if backbone == "densenet121" and arm == "A2_bg":
        return base / "bg_suppression_followup" / "runs" / "A2_bg"
    if seed == 42:
        return base / "runs" / arm
    if backbone == "efficientnet_b0":
        return base / "runs_multiseed" / f"seed_{seed}" / "runs" / arm
    return base / "runs_multiseed" / f"{arm}_seed{seed}"


def discover_checkpoints(root):
    rows = []
    for d, subdirs, files in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        for name in files:
            path = Path(d) / name
            if not name.endswith(".pt") or "smoke" in str(path).lower():
                continue
            ck = torch_load(path)
            if not (isinstance(ck, dict) and "model_state_dict" in ck and "config" in ck):
                print("skipped (not a project checkpoint):", path)
                continue
            seed = int(ck.get("seed", 42))
            arm = str(ck.get("arm", path.stem))
            if arm.endswith(f"_seed{seed}"):
                arm = arm[: -len(f"_seed{seed}")]
            model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
            backbone = model_cfg["name"]
            if backbone not in ARTIFACT:
                print("skipped (unknown backbone):", path)
                continue
            rows.append({
                "backbone": backbone, "arm": arm, "seed": seed,
                "use_attention": module_cfg.get("use_attention"), "attention": module_cfg.get("attention", "lung"),
                "gate_mode": module_cfg.get("gate_mode"), "lambda_att": module_cfg.get("lambda_att"),
                "lambda_bg": module_cfg.get("lambda_bg"),
                "size_mb": round(path.stat().st_size / 2**20, 1), "path": str(path),
            })
            del ck
    inv = pd.DataFrame(rows)
    if inv.empty:
        raise FileNotFoundError(f"no .pt checkpoints found under {root}. Attach the checkpoint datasets.")
    before = len(inv)
    inv = inv.sort_values(["backbone", "seed", "arm", "path"]).drop_duplicates(["backbone", "arm", "seed"])
    if len(inv) < before:
        print(f"{before - len(inv)} duplicate checkpoint files ignored (same backbone/arm/seed)")
    inv["role"] = [
        "baseline" if a == BASE_ARM[b] else "selected" if a == SELECTED_ARM[b] else "other"
        for b, a in zip(inv.backbone, inv.arm)]
    inv["run"] = inv.backbone + "/" + inv.arm + "/seed_" + inv.seed.astype(str)
    return inv.reset_index(drop=True)


DATA_DIR = find_data_dir(DATA_ROOT)
print("dataset:", DATA_DIR)
INVENTORY = discover_checkpoints(CKPT_ROOT)
print(f"{len(INVENTORY)} checkpoints found")
print(INVENTORY.groupby(["backbone", "seed"]).size().unstack(fill_value=0))

# What the core grid needs: baseline + selected arm at three seeds for every backbone (24 checkpoints).
have = set(zip(INVENTORY.backbone, INVENTORY.arm, INVENTORY.seed))
missing_core = [(b, a, s) for b in ARTIFACT for a in (BASE_ARM[b], SELECTED_ARM[b]) for s in SEEDS
                if (b, a, s) not in have]
print("core grid checkpoints missing:", len(missing_core))
for m in missing_core:
    print("   MISSING", m)

In [ ]:
import platform
from notebooks.efficiency import benchmark_model
from src.modules import LogitsOnly, build_model

DISPLAY = {"densenet121": "DenseNet121", "resnet50": "ResNet50", "efficientnet_b0": "EfficientNet-B0",
           "vit_base_patch16_224": "ViT-Base/16"}


def load_cpu_model(row):
    ck = torch_load(row["path"])
    model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
    model = build_model(
        num_classes=model_cfg["num_classes"], use_attention=module_cfg["use_attention"],
        reduction=module_cfg.get("reduction", 8), gate_mode=module_cfg.get("gate_mode", "residual"),
        backbone_name=model_cfg["name"], pretrained=False,
        attention=module_cfg.get("attention", "lung"), drop_rate=model_cfg.get("drop_rate", 0.0))
    model.load_state_dict(ck["model_state_dict"], strict=True)
    return LogitsOnly(model).eval()


@torch.inference_mode()
def gpu_batch_throughput(model, batch=32, warmup=10, runs=30):
    if DEVICE.type != "cuda":
        return None
    model = model.to(DEVICE)
    x = torch.randn(batch, 3, 224, 224, device=DEVICE)
    for _ in range(warmup):
        model(x)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        model(x)
    torch.cuda.synchronize()
    return batch * runs / (time.perf_counter() - start)


grid = INVENTORY[INVENTORY.seed == 42]
if not ALL_SEED42_ARMS:
    grid = grid[grid.role.isin(["baseline", "selected"])]
grid = grid.sort_values(["backbone", "arm"])
if DRY_RUN:
    grid = grid.head(2)
print(len(grid), "models to benchmark")

rows = []
for n, row in enumerate(grid.to_dict("records"), 1):
    t1 = time.time()
    model = load_cpu_model(row)
    result = benchmark_model(model, DISPLAY[row["backbone"]], row["arm"], checkpoint_path=row["path"],
                             cpu_runs=CPU_RUNS, gpu_runs=GPU_RUNS)
    result["gpu_throughput_batch32_img_s"] = gpu_batch_throughput(model)
    result.update(backbone=row["backbone"], arm=row["arm"], role=row["role"], seed=row["seed"])
    rows.append(result)
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"[{n}/{len(grid)}] {row['run']} done in {time.time() - t1:.0f}s")

eff = pd.DataFrame(rows)
front = ["backbone", "arm", "role", "seed"]
eff = eff[front + [c for c in eff.columns if c not in front]]
eff.to_csv(OUT / "efficiency_all.csv", index=False)
if eff["gflops"].isna().any():
    msg = "GFLOPs is NaN: fvcore did not install. Turn Internet on and Run All again."
    if DRY_RUN:
        print("WARNING:", msg)
    else:
        raise RuntimeError(msg)

In [ ]:
hardware = {
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "cpu": platform.processor() or platform.machine(), "cpu_count": os.cpu_count(),
    "platform": platform.platform(), "python": platform.python_version(), "torch": torch.__version__,
    "input_shape": [1, 3, 224, 224], "cpu_runs": CPU_RUNS, "gpu_runs": GPU_RUNS, "repo_commit": COMMIT,
    "note": "All rows were measured in this one session. ViT GFLOPs are a partial estimate: fvcore does not "
            "count fused attention operations.",
}
try:
    hardware["cpu_model"] = next(l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except Exception:
    pass
(OUT / "efficiency_hardware.json").write_text(json.dumps(hardware, indent=2))
print(json.dumps(hardware, indent=2))

# Cost of the module: every arm against its backbone's A0. Budget from the acceptance criteria:
# parameters +3.0% at most, GFLOPs +2.0% at most.
over = []
for backbone, g in eff.groupby("backbone"):
    base = g[g.role == "baseline"]
    if base.empty:
        continue
    b = base.iloc[0]
    for r in g[g.role != "baseline"].itertuples():
        over.append({
            "backbone": backbone, "arm": r.arm,
            "params_delta_pct": 100 * (r.params_total / b.params_total - 1),
            "gflops_delta_pct": 100 * (r.gflops / b.gflops - 1),
            "cpu_latency_delta_pct": 100 * (r.cpu_latency_ms / b.cpu_latency_ms - 1),
            "gpu_latency_delta_pct": 100 * (r.gpu_latency_ms / b.gpu_latency_ms - 1) if b.gpu_latency_ms else None,
            "within_budget": bool(100 * (r.params_total / b.params_total - 1) <= 3.0
                                  and 100 * (r.gflops / b.gflops - 1) <= 2.0),
        })
overhead = pd.DataFrame(over)
overhead.to_csv(OUT / "efficiency_overhead.csv", index=False)
pd.set_option("display.width", 220)
print(eff[["backbone", "arm", "params_total", "gflops", "checkpoint_size_mb", "cpu_latency_ms", "gpu_latency_ms",
           "gpu_throughput_img_s", "gpu_throughput_batch32_img_s"]].round(3).to_string(index=False))
print()
print(overhead.round(3).to_string(index=False))

In [ ]:
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Download this file from the Output panel:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MB)")